# A block as a sum of terms

After `reduce_gpf`, a block is a small diagram in which every input has its own Z-spider and every output its own X-spider. The other spiders are *hidden*: a hidden Z-spider is a bit that is summed over, a hidden X-spider a parity check. Setting each hidden spider to 0 and to 1 writes the block as a sum of simpler operations, its *terms* (L. Yeh's decomposition). `decompose_terms` does this and says what each term does: which single-qubit gate it applies to each qubit, or which qubits it links, and what each single-qubit gate does to an X, Y or Z error.

Two things to know:

* **The terms depend on how the hidden bits are labelled.** A hidden bit `z` can be relabelled as `z + (some inputs)`: the block stays the same, the terms change. `labelling='as reduced'` keeps the labels `reduce_gpf` chose; `labelling='fewest links'` adds inputs to the hidden bits so that each input reaches as few outputs as possible.
* **The block is split into connected parts first.** The terms of the block are products of the terms of its parts, so each part is split on its own. A part with too many terms (more than `max_terms`, 1024 by default) is reported but not split.

The check for distance (agreed with L. Yeh): **in every part, every term maps each Pauli error to the same Pauli**, up to a sign (`same_error_action`). It looks at what each term does to an X, Y or Z error on each input, for terms that link qubits too. Products of single-qubit Clifford gates are not enough: below, the d = 3 double-check splits into such terms and still loses distance with real T. The check is not a proof of distance; it agrees with the known fault distances of the two versions below.

Whether the terms are single-qubit is reported too, but it is only expected of the non-Clifford parts: a Clifford part generally has terms that link qubits, as one CNOT does. The table marks each part Clifford or non-Clifford.

In [1]:
import matplotlib
matplotlib.use('Agg')
from zxqec.qprogram import program_from_stim
from zxqec.gpf import reduce_gpf, boolean_variables, consistent_assignments, decompose_terms



D3 = open('circuits/cultivation/d3_cultivation.stim').read()
GROUPS_D3 = [(0, 0, 'first checks', 'state preparation'), (1, 1, 'inject T', 'injection'),
             (2, 2, 'checks', 'syndrome extraction'), (3, 4, 'double-check', 'cultivation')]
p = program_from_stim(D3)[0]
g = p.block_graph(p.group_stages(GROUPS_D3), 3, classical=False)      # the double-check, a block of the program
res = reduce_gpf(g)
runs = consistent_assignments(res.constraints, sorted(set(boolean_variables(g)) |
                                                        {str(v) for c in res.constraints for v in c.free_vars()}))
print(len(runs), "runs of the double-check are allowed by its detectors; the first:", runs[0])

4 runs of the double-check are allowed by its detectors; the first: {'c[24]': 0, 'c[25]': 0, 'c[26]': 0, 'c[27]': 0, 'c[28]': 0, 'c[29]': 0, 'c[30]': 0, 's3_r0': 0}


## The double-check of d = 3 cultivation

The double-check applies T† to the seven code qubits, measures the logical qubit with helper qubits, and applies T. As `reduce_gpf` labels its one hidden bit, each term links the code qubits (part 1); part 2 is helper qubit 1's old state, which the block measures and throws away:

In [2]:
print(decompose_terms(res, runs[0], 'as reduced'))

labelling 'as reduced': 2 part(s), 2 hidden spider(s)
part 1 (non-Clifford; inputs [0, 3, 6, 8, 9, 11, 14], outputs [0, 3, 6, 8, 9, 11, 14]): hidden Z-spider z1; 2 terms, 2 non-zero
  term 0:
      links qubits 0, 3, 6, 8, 9, 11, 14 (inputs [0, 3, 6, 8, 9, 11, 14], outputs [0, 3, 6, 8, 9, 11])
      qubit 14: |0>
  term 1:
      links qubits 0, 3, 6, 8, 9, 11, 14 (inputs [0, 3, 6, 8, 9, 11, 14], outputs [0, 3, 6, 8, 9, 11])
      qubit 14: |1>
part 2 (Clifford; inputs [1], outputs []): hidden X-spider x1; 2 terms, 2 non-zero
  term 0:
      qubit 1: <+|
  term 1:
      qubit 1: <-|
in every part, every term maps each Pauli error to the same Pauli (up to a sign): None
every term of the non-Clifford parts single-qubit: False (every term single-qubit: False)


As reduced, each term on its own is a projection, not a unitary, so it maps no Pauli to a Pauli and the check has no answer (`None`). With the hidden bit relabelled as `z + input 14`, the two terms are: do nothing, and T X T† on every code qubit. The brackets say what each gate does to an error on its qubit: T X T† turns an X error into a Y error while the other term keeps it, so the check fails.

In [3]:
print(decompose_terms(res, runs[0], 'fewest links'))

labelling 'fewest links': 2 part(s), 2 hidden spider(s)
part 1 (non-Clifford; inputs [0, 3, 6, 8, 9, 11, 14], outputs [0, 3, 6, 8, 9, 11, 14]): hidden Z-spider z1 (+ inputs 14); 2 terms, 2 non-zero
  term 0:
      qubit 0: I (X->X, Y->Y, Z->Z)
      qubit 3: I (X->X, Y->Y, Z->Z)
      qubit 6: I (X->X, Y->Y, Z->Z)
      qubit 8: I (X->X, Y->Y, Z->Z)
      qubit 9: I (X->X, Y->Y, Z->Z)
      qubit 11: I (X->X, Y->Y, Z->Z)
      qubit 14: I (X->X, Y->Y, Z->Z)
  term 1:
      qubit 0: T X T^dag (X->Y, Y->X, Z->Z)
      qubit 3: T X T^dag (X->Y, Y->X, Z->Z)
      qubit 6: T X T^dag (X->Y, Y->X, Z->Z)
      qubit 8: T X T^dag (X->Y, Y->X, Z->Z)
      qubit 9: T X T^dag (X->Y, Y->X, Z->Z)
      qubit 11: T X T^dag (X->Y, Y->X, Z->Z)
      qubit 14: T X T^dag (X->Y, Y->X, Z->Z)
part 2 (Clifford; inputs [1], outputs []): hidden X-spider x1; 2 terms, 2 non-zero
  term 0:
      qubit 1: <+|
  term 1:
      qubit 1: <-|
in every part, every term maps each Pauli error to the same Pauli (up to a si

## The same with S in place of T

Gidney et al. simulate cultivation with S and S† in place of T and T†, a version whose fault distance is known to be 3 (with real T gates it is 2: Chan et al., arXiv:2609.17706). Here the second term is Y on every qubit, which leaves every error as it is (up to a sign):

In [4]:
swap = {'T': 'S', 'T_DAG': 'S_DAG'}
D3_S = '\n'.join(swap[l.split(' ', 1)[0]] + ' ' + l.split(' ', 1)[1] if l.split(' ', 1)[0] in swap else l
                  for l in D3.splitlines())
p_s = program_from_stim(D3_S)[0]
g_s = p_s.block_graph(p_s.group_stages(GROUPS_D3), 3, classical=False)
res_s = reduce_gpf(g_s)
runs_s = consistent_assignments(res_s.constraints, sorted(set(boolean_variables(g_s)) |
                                                            {str(v) for c in res_s.constraints for v in c.free_vars()}))
print(decompose_terms(res_s, runs_s[0], 'fewest links'))

labelling 'fewest links': 2 part(s), 2 hidden spider(s)
part 1 (Clifford; inputs [0, 3, 6, 8, 9, 11, 14], outputs [0, 3, 6, 8, 9, 11, 14]): hidden Z-spider z1 (+ inputs 14); 2 terms, 2 non-zero
  term 0:
      qubit 0: I (X->X, Y->Y, Z->Z)
      qubit 3: I (X->X, Y->Y, Z->Z)
      qubit 6: I (X->X, Y->Y, Z->Z)
      qubit 8: I (X->X, Y->Y, Z->Z)
      qubit 9: I (X->X, Y->Y, Z->Z)
      qubit 11: I (X->X, Y->Y, Z->Z)
      qubit 14: I (X->X, Y->Y, Z->Z)
  term 1:
      qubit 0: Y (X->X, Y->Y, Z->Z)
      qubit 3: Y (X->X, Y->Y, Z->Z)
      qubit 6: Y (X->X, Y->Y, Z->Z)
      qubit 8: Y (X->X, Y->Y, Z->Z)
      qubit 9: Y (X->X, Y->Y, Z->Z)
      qubit 11: Y (X->X, Y->Y, Z->Z)
      qubit 14: Y (X->X, Y->Y, Z->Z)
part 2 (Clifford; inputs [1], outputs []): hidden X-spider x1; 2 terms, 2 non-zero
  term 0:
      qubit 1: <+|
  term 1:
      qubit 1: <-|
in every part, every term maps each Pauli error to the same Pauli (up to a sign): True
every term of the non-Clifford parts single-qubit:

Side by side, over all runs the detectors allow:

In [5]:
for name, (r, rs) in (('real T', (res, runs)), ('S for T', (res_s, runs_s))):
    for labelling in ('as reduced', 'fewest links'):
        ds = [decompose_terms(r, run, labelling) for run in rs]
        print("{:8} {:13} every term maps each Pauli to the same Pauli: {:14}  non-Clifford parts single-qubit: {}".format(
            name, labelling, str(sorted({str(d.same_error_action) for d in ds})),
            sorted({str(d.non_clifford_single_qubit) for d in ds})))

real T   as reduced    every term maps each Pauli to the same Pauli: ['None']        non-Clifford parts single-qubit: ['False']
real T   fewest links  every term maps each Pauli to the same Pauli: ['False']       non-Clifford parts single-qubit: ['True']
S for T  as reduced    every term maps each Pauli to the same Pauli: ['None']        non-Clifford parts single-qubit: ['None']


S for T  fewest links  every term maps each Pauli to the same Pauli: ['True']        non-Clifford parts single-qubit: ['None']

With real T the non-Clifford part's terms are single-qubit, yet the check fails; with S there is no non-Clifford part and the check passes, as the fault distances (2 and 3) say. With the labels of the reduction the check has no answer in either version.

## Other blocks

The fold-transversal Hadamard (Clifford): the transversal H, the 90° rotation and a round of checks.

In [6]:
import interactive_fold_h_3d as fth
FOLD_GROUPS = [(0, 0, 'encode + round 1', 'encoder'), (1, 1, 'transversal H', 'logical gate'),
               (2, 3, 'round in code 2', 'syndrome extraction'), (4, 4, 'rotate', 'logical gate'),
               (5, 6, 'round in code 1', 'syndrome extraction'), (7, 7, 'checks + decode', 'decoder')]
fold = fth.build_program()
level = fold.group_stages(FOLD_GROUPS)
for index in (1, 3):
    r = reduce_gpf(fold.block_graph(level, index, classical=False))       # Clifford, no outcomes left: no run needed
    print(FOLD_GROUPS[index][2])
    print(decompose_terms(r).table())
    print()

transversal H
labelling 'as reduced': 14 part(s), 0 hidden spider(s)
part 1 (Clifford; inputs [1], outputs [1]): no hidden spiders: qubit 1: H (X->Z, Y->Y, Z->X)
part 2 (Clifford; inputs [2], outputs [2]): no hidden spiders: qubit 2: H (X->Z, Y->Y, Z->X)
part 3 (Clifford; inputs [3], outputs [3]): no hidden spiders: qubit 3: H (X->Z, Y->Y, Z->X)
part 4 (Clifford; inputs [4], outputs [4]): no hidden spiders: qubit 4: H (X->Z, Y->Y, Z->X)
part 5 (Clifford; inputs [5], outputs [5]): no hidden spiders: qubit 5: H (X->Z, Y->Y, Z->X)
part 6 (Clifford; inputs [6], outputs [6]): no hidden spiders: qubit 6: H (X->Z, Y->Y, Z->X)
part 7 (Clifford; inputs [7], outputs [7]): no hidden spiders: qubit 7: H (X->Z, Y->Y, Z->X)
part 8 (Clifford; inputs [8], outputs [8]): no hidden spiders: qubit 8: H (X->Z, Y->Y, Z->X)
part 9 (Clifford; inputs [9], outputs [9]): no hidden spiders: qubit 9: H (X->Z, Y->Y, Z->X)
part 10 (Clifford; inputs [14], outputs [14]): no hidden spiders: qubit 14: I (X->X, Y->Y, Z->

In [7]:
gb = fold.block_graph(level, 4, classical=False)
r = reduce_gpf(gb)
first_run = consistent_assignments(r.constraints, sorted(set(boolean_variables(gb)) |
                                                         {str(v) for c in r.constraints for v in c.free_vars()}))[0]
d = decompose_terms(r, first_run, 'fewest links')
print("round in code 1:", len(d.parts), "parts,", [len(p.hidden) for p in d.parts if p.hidden], "hidden spiders,",
      "every term single-qubit:", d.single_qubit, "| every term maps each Pauli to the same Pauli:", d.same_error_action)
print('\n'.join(d.table().splitlines()[:14]))

round in code 1: 2 parts, [8] hidden spiders, every term single-qubit: False | every term maps each Pauli to the same Pauli: None
labelling 'fewest links': 2 part(s), 8 hidden spider(s)
part 1 (Clifford; inputs [1, 2, 3, 4, 5, 6, 7, 8, 9], outputs [1, 2, 3, 4, 5, 6, 7, 8, 9]): hidden Z-spider z1; Z-spider z2; Z-spider z3; Z-spider z4; X-spider x1; X-spider x2; X-spider x3; X-spider x4; 256 terms, 256 non-zero
  term 00000000:
      links qubits 1, 2, 3, 6, 9 (inputs [3, 6, 9], outputs [1, 2, 3])
      qubit 1: <+|
      qubit 2: <+|
      qubit 4: <+|
      qubit 4: |0>
      qubit 5: <+|
      qubit 5: |0>
      qubit 6: |0>
      qubit 7: <+|
      qubit 7: |0>
      qubit 8: <+|


The transversal H reads as H on every data qubit. The rotation is the identity on every program qubit: a permutation stage only moves qubits to other physical positions. A round of checks has terms that link data and helper qubits: it is Clifford, and a CNOT already links two qubits, so single-qubit terms are not expected there.

d = 5 cultivation: the two double-checks (the d = 3 one inside the d = 5 protocol, and the d = 5 one) behave like the d = 3 double-check above. The rounds of checks at d = 5 have a part with 18 hidden spiders, 2¹⁸ terms, which is reported but not split.

In [8]:
GROUPS_D5 = [(0, 0, 'inject T', 'injection'), (1, 2, 'double-check d=3', 'cultivation'), (3, 3, 'grow', None),
             (4, 5, 'checks', 'syndrome extraction'), (6, 7, 'double-check d=5', 'cultivation')]
d5 = program_from_stim(open('circuits/cultivation/d5_cultivation.stim').read())[0]
level5 = d5.group_stages(GROUPS_D5)
for index in range(5):
    gb = d5.block_graph(level5, index, classical=False)
    r = reduce_gpf(gb)
    first_run = consistent_assignments(r.constraints, sorted(set(boolean_variables(gb)) |
                                                             {str(v) for c in r.constraints for v in c.free_vars()}))[0]
    for labelling in ('as reduced', 'fewest links'):
        d = decompose_terms(r, first_run, labelling)
        print("{:17} {:13} parts {:3}  not split {}  same Pauli {!s:5}  non-Clifford parts single-qubit {}".format(
            GROUPS_D5[index][2], labelling, len(d.parts), [len(p.hidden) for p in d.parts if p.terms is None],
            d.same_error_action, d.non_clifford_single_qubit))

inject T          as reduced    parts  29  not split []  same Pauli True   non-Clifford parts single-qubit True
inject T          fewest links  parts  29  not split []  same Pauli True   non-Clifford parts single-qubit True


double-check d=3  as reduced    parts  35  not split []  same Pauli None   non-Clifford parts single-qubit False
double-check d=3  fewest links  parts  35  not split []  same Pauli False  non-Clifford parts single-qubit True


grow              as reduced    parts  11  not split [18]  same Pauli None   non-Clifford parts single-qubit None
grow              fewest links  parts  11  not split [18]  same Pauli None   non-Clifford parts single-qubit None


checks            as reduced    parts   4  not split [18]  same Pauli None   non-Clifford parts single-qubit None
checks            fewest links  parts   4  not split [18]  same Pauli None   non-Clifford parts single-qubit None
double-check d=5  as reduced    parts   4  not split []  same Pauli None   non-Clifford parts single-qubit False
double-check d=5  fewest links  parts   4  not split []  same Pauli False  non-Clifford parts single-qubit True


## What this is not

* **Not a proof of distance.** The check (every term maps each Pauli to the same Pauli) agrees with the known fault distances of the d = 3 double-check, 3 with S and 2 with T; it is not proved to decide the distance. It has no answer when a term is not a unitary (as with the labels of the reduction) or a part is too large to split.
* **One relabelling rule.** `'fewest links'` minimises, input by input, how many outputs the input's Z-spider reaches; other rules may suit other protocols. Only hidden Z-spiders are relabelled.
* **Small parts only.** A part with `k` hidden spiders has `2^k` terms; parts above `max_terms` are not split.